# Sıfırdan Dil Modeli · Bölüm 7
## Dikkat III: maske ve çok başlı dikkat

Geçen bölümde dikkate üç öğrenilebilir matris verdik: sorgu, anahtar, değer (`SelfAttention_v2`). Ama o dikkatte her kelime cümlenin **bütün** kelimelerine bakıyor, sonrakilere de. Metni soldan sağa yazan bir model için bu kopya çekmek demek. Bu defterde önce geleceği bir **maskeyle** kapatıyoruz, sonra dikkati **çok başlı** hâle getiriyoruz: aynı anda birkaç ayrı dikkat. En sonda serinin modeli Çırak'ın gerçek dikkat katmanını kendi sınıfımıza yüklüyoruz.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 3.5–3.6. bölümlerini izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. Maske
Önceki bölümlerin kodu `onceki_bolumler.py` dosyasında. Onu serinin deposundan indiriyoruz.

In [ ]:
import os
import urllib.request
import torch
import torch.nn as nn

DEPO = "https://raw.githubusercontent.com/kocahmet1/sifirdan-dil-modeli/main/"

def indir(path):
    dosya = os.path.basename(path)
    if not os.path.exists(dosya):
        urllib.request.urlretrieve(DEPO + path, dosya)
    print(dosya, os.path.getsize(dosya) // 1024, "KB")

indir("onceki_bolumler.py")

Geçen bölümün bittiği yer: Bölüm 5'in cümlesi ve altı vektörü, `SelfAttention_v2` ve kitaptaki tohum (789). Ağırlıklar Bölüm 6'dakilerin aynısı.

In [ ]:
from onceki_bolumler import SelfAttention_v2, isi_haritasi

kelimeler = "Babam yalağın içinde kırılmış kaşağıyı gördü".split()
inputs = torch.tensor(
  [[0.90, 0.10, 0.20],   # Babam
   [0.10, 0.40, 0.80],   # yalağın
   [0.10, 0.20, 0.70],   # içinde
   [0.20, 0.90, 0.30],   # kırılmış
   [0.10, 0.80, 0.30],   # kaşağıyı
   [0.70, 0.30, 0.20]]   # gördü
)
d_in, d_out = 3, 2
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2)

Dikkat ağırlıkları, `SelfAttention_v2`'nin içindeki hesabın aynısı: sorgular çarpı anahtarlar, √d_k'ya böl, satır satır softmax.

In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

In [ ]:
isi_haritasi(attn_weights, kelimeler)

Her satır bir kelimenin nereye baktığı. İlk satır Babam: cümlenin sonuna, gördü'ye de bakıyor. Ama model metni soldan sağa yazar: Babam'ı yazarken gördü daha yok. Her kelime yalnızca kendine ve kendinden öncekilere bakabilmeli. `torch.tril` bir matrisin alt üçgenini bırakır (*lower triangle*), üstünü sıfırlar.

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

In [ ]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

Köşegenin üstü sıfır. Ama satır toplamları artık 1 değil: her satırı kendi toplamına bölüyoruz. `keepdim=True`, toplamın satır şeklini korur; bölme her satıra kendi toplamıyla yapılır.

In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

## 2. Eksi sonsuz
Aynı sonuca tek adımda: maskeyi softmax'tan **önce**, skorlara uygulamak. Gizlenecek skorların yerine eksi sonsuz yazıyoruz; e üzeri eksi sonsuz sıfırdır, softmax onlara hiç pay vermez. `torch.triu(..., diagonal=1)` köşegenin üstünü 1 yapar: gizlenecek yerler. `masked_fill` o yerlere verilen değeri yazar.

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

In [ ]:
isi_haritasi(attn_weights, kelimeler)

## 3. Dropout
Eğitim sırasında rastgele bazı ağırlıkları sıfırlamak: model tek bir yola fazla güvenmesin, ezberlemesin. Yarısı sıfırlanınca kalanlar 2 ile çarpılır (1 / 0,5), toplam ölçek değişmesin diye. Yalnızca eğitimde çalışır; model kullanılırken kapalıdır.

In [ ]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)            # %50
example = torch.ones(6, 6)
print(dropout(example))

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

## 4. CausalAttention
Maskeyi ve dropout'u `SelfAttention_v2`'ye ekleyip yeni bir sınıf yazıyoruz. Bir fark daha: model cümleleri paket paket (*batch*) alır. Deneme için aynı cümleyi iki kez üst üste koyuyoruz: `torch.stack` tensörleri yeni bir boyutta üst üste dizer.

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)                    # yeni
        self.register_buffer("mask", torch.triu(              # yeni
            torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape                         # yeni: paket
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)          # .T yerine
        attn_scores.masked_fill_(                             # yeni: maske
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)             # yeni

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print(context_vecs.shape)

## 5. Çok başlı dikkat
Bir dikkat, cümleye tek bir açıdan bakar. Birkaç `CausalAttention`'ı yan yana çalıştırıp çıktılarını uç uca ekleyelim: her birine **başlık** (*head*) denir. `nn.ModuleList` katmanları bir listede tutar; köşeli parantezin içindeki `for`, listeyi tek satırda kurar. `torch.cat` tensörleri uç uca ekler.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
torch.manual_seed(123)
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs.shape)

## 6. MultiHeadAttention
Sarmalayıcı başlıkları sırayla çalıştırır. Gerçek modeller tek büyük bir sorgu, anahtar ve değer matrisi kullanır, sonucu başlıklara böler (`view`), başlık boyutunu öne alır (`transpose`) ve bütün başlıkları tek bir matris çarpımında hesaplar. En sonda başlıklar yeniden birleşir ve bir çıkış katmanı (`out_proj`) onları karıştırır.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out, num_heads'e tam bölünmeli"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads          # başlık başına boyut
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)     # başlıkları karıştırır
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(
            torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)                        # (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # son boyutu başlıklara böl: (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # başlıkları öne al: (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)   # her başlık ayrı
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        self.attn_weights = attn_weights   # sonra bakmak için (kitapta yok)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
torch.manual_seed(123)
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs.shape)
print(mha.attn_weights.shape)

Aynı sınıf, GPT-2'nin boyutunda: 768 sayılık vektörler, 1.024 token'lık bağlam, 12 başlık. Kaç parametre? `numel` bir tensördeki sayı adedini verir.

In [ ]:
mha_gpt2 = MultiHeadAttention(d_in=768, d_out=768, context_length=1024,
                              dropout=0.0, num_heads=12, qkv_bias=True)
print(sum(p.numel() for p in mha_gpt2.parameters()))

## 7. Çırak'ın başlıkları
Kendi sınıfımıza gerçek ağırlıklar yükleyelim: serinin modeli **Çırak**'ın (Türkçe Vikipedi'de eğitildi) 2. bloğundaki dikkat katmanı. 8 başlık, 256 sayılık vektörler. Bu katmana giren vektörleri bu cümle için hazır veriyoruz: onları üreten ilk blok, Bölüm 8–9'da yazacağımız parçalardan oluşuyor.

In [ ]:
indir("veri/cirak_blok2.pt")
cirak = torch.load("cirak_blok2.pt")
print(cirak["cumle"])
print(len(cirak["parcalar"]), cirak["parcalar"])
print(cirak["girdi"].shape)

In [ ]:
mha_cirak = MultiHeadAttention(d_in=256, d_out=256, context_length=256,
                               dropout=0.0, num_heads=8, qkv_bias=True)
print(mha_cirak.load_state_dict(cirak["dikkat"]))

In [ ]:
with torch.no_grad():
    out = mha_cirak(cirak["girdi"].unsqueeze(0))
A = mha_cirak.attn_weights[0]                # 8 başlık, her biri 15 × 15
print(out.shape, A.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for h, ax in enumerate(axes.flat):
    ax.imshow(A[h], cmap="viridis")
    ax.set_title(f"başlık {h + 1}")
    ax.axis("off")
plt.show()

Her başlığın kendi deseni var. Birinci başlığa yakından bakalım:

In [ ]:
isi_haritasi(A[0], cirak["parcalar"])

## Özet
Maske, her kelimenin yalnızca kendine ve öncekilere bakmasını sağlar: skorlarda köşegenin üstü eksi sonsuz, softmax'tan sonra sıfır. Dropout eğitimde rastgele ağırlıkları sıfırlar. `CausalAttention` ikisini paketlerle birlikte tek sınıfta toplar; `MultiHeadAttention` birkaç başlığı tek bir matris çarpımında çalıştırır. Çırak'ın gerçek başlıklarında her başlık cümleye başka türlü bakıyor: biri her parçayı kelimesinin başına baktırıyor, biri cümlenin ilk parçasına.

**Kendin dene**
- `dropout=0.5` ile `CausalAttention`'ı iki kez çalıştırın: çıktılar neden farklı?
- Çırak'ın 5. başlığına `isi_haritasi(A[4], cirak["parcalar"])` ile yakından bakın: kelimelerin ilk parçaları nereye bakıyor?

**Sıradaki bölüm:** GPT'nin parçaları: katman normu, GELU, ileri besleme, kısa yollar.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 3.5–3.6. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Cümle: Ömer Seyfettin, "Kaşağı" (kamu malı). Çırak'ın ağırlıkları ve tokenizer'ı Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.